# Validation on In-the-Wild (3k real / 3k fake)
Runs the full augmentation attack sweep on a **balanced 6 000-sample** subset of `mueller91/In-The-Wild`.

## 0. Setup & Torch patches

In [5]:
import torch
from argparse import Namespace
from ay2.tools.text._phonemes import Phonemer_Tokenizer_Recombination
from pandas import Series

torch.serialization.add_safe_globals([
    Namespace,
    Phonemer_Tokenizer_Recombination,
    Series,
])

In [6]:
_orig_torch_load = torch.load
def _torch_load_no_weights_only(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_torch_load(*args, **kwargs)
torch.load = _torch_load_no_weights_only
print('Patched torch.load -> weights_only=False')

Patched torch.load -> weights_only=False


## 1. Load In-the-Wild — balanced 3k real / 3k fake

In [7]:
import os, sys, random
notebook_dir = os.getcwd()
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

def load_hf_token(path='secret.txt'):
    if os.path.exists(path):
        with open(path) as f:
            return f.read().strip()
    return None

HF_TOKEN = load_hf_token()
print('HF token:', 'YES' if HF_TOKEN else 'NO (set secret.txt if needed)')

HF token: YES


In [8]:
from datasets import load_dataset, Audio
from collections import Counter
from huggingface_hub import hf_hub_download
import pandas as pd

ITW_CACHE   = './data/in_the_wild'
N_PER_CLASS = 3000  # 3k real + 3k fake = 6k total

meta = pd.read_csv("meta.csv")
print('meta.csv shape:', meta.shape)
print('meta.csv columns:', meta.columns.tolist())
print(meta.head())
print('\nLabel value counts:')
print(meta.iloc[:, 1].value_counts())   # second column = label


meta.csv shape: (31779, 3)
meta.csv columns: ['file', 'speaker', 'label']
    file               speaker      label
0  0.wav         Alec Guinness      spoof
1  1.wav         Alec Guinness      spoof
2  2.wav          Barack Obama      spoof
3  3.wav         Alec Guinness      spoof
4  4.wav  Christopher Hitchens  bona-fide

Label value counts:
speaker
Barack Obama                3636
Alec Guinness               3625
Donald Trump                3423
Bernie Sanders              2877
Ayn Rand                    2493
Bill Clinton                1832
Ronald Reagan               1536
Christopher Hitchens        1339
Winston Churchill            882
Martin Luther King           799
JFK                          669
Milton Friedman              589
Mark Zuckerberg              582
FDR                          471
Queen Elizabeth II           464
Louis Farrakhan              410
Alexandria Ocasio-Cortez     390
Nelson Mandela               381
Alan Watts                   378
Richard Nixon     

In [9]:
# ── Step 2: load audio from HF (streaming cache) ───────────────────────────
ds_itw = load_dataset(
    'mueller91/In-The-Wild',
    cache_dir=ITW_CACHE,
    token=HF_TOKEN if HF_TOKEN else None,
)
split_name   = list(ds_itw.keys())[0]
ds_itw_split = ds_itw[split_name].cast_column('audio', Audio(sampling_rate=16000))
print(f'HF split: {split_name!r}  |  samples: {len(ds_itw_split)}')

# ── Step 3: extract basename from HF audio path and join to meta.csv ─────────
# meta.csv uses column names like 'file' and 'label'  (confirmed from researchers)
# Normalise column names defensively
meta.columns = [c.strip().lower() for c in meta.columns]

# Identify file-name and label columns
file_col  = next(c for c in meta.columns if 'file' in c or 'path' in c or 'name' in c)
label_col = next(c for c in meta.columns if c != file_col)
print(f'\nUsing  file_col={file_col!r}  label_col={label_col!r}')

# Build a lookup: basename (no extension) -> label string
import os
meta['_basename'] = meta[file_col].apply(lambda p: os.path.splitext(os.path.basename(str(p)))[0])
basename_to_label = dict(zip(meta['_basename'], meta[label_col]))
print(f'Lookup entries: {len(basename_to_label)}')

# Read basenames from HF dataset using decode=False (no waveform loading)
ds_nodecode = ds_itw_split.cast_column('audio', Audio(decode=False))
hf_basenames = [
    os.path.splitext(os.path.basename(ds_nodecode[i]['audio']['path']))[0]
    for i in range(len(ds_nodecode))
]

# Join: get a label string for every HF row (None if not in meta)
raw_label_strs = [basename_to_label.get(b) for b in hf_basenames]
missing = sum(1 for l in raw_label_strs if l is None)
print(f'Matched: {len(raw_label_strs) - missing}/{len(raw_label_strs)}  |  Missing: {missing}')
print('Label counts (raw):', Counter(l for l in raw_label_strs if l is not None))


HF split: 'train'  |  samples: 31069

Using  file_col='file'  label_col='speaker'
Lookup entries: 31779
Matched: 31069/31069  |  Missing: 0
Label counts (raw): Counter({'Barack Obama': 3558, 'Alec Guinness': 3544, 'Donald Trump': 3347, 'Bernie Sanders': 2811, 'Ayn Rand': 2442, 'Bill Clinton': 1790, 'Ronald Reagan': 1509, 'Christopher Hitchens': 1308, 'Winston Churchill': 857, 'Martin Luther King': 782, 'JFK': 659, 'Milton Friedman': 577, 'Mark Zuckerberg': 566, 'FDR': 460, 'Queen Elizabeth II': 453, 'Louis Farrakhan': 399, 'Alexandria Ocasio-Cortez': 384, 'Nelson Mandela': 367, 'Alan Watts': 365, 'Richard Nixon': 354, 'Arnold Schwarzenegger': 341, 'The Notorious B.I.G.': 321, 'Boris Johnson': 289, 'Dwight Eisenhower': 262, 'George Carlin': 241, 'George W. Bush': 226, 'Adam Driver': 215, 'Gilbert Gottfried': 203, 'Bill Burr': 199, 'Orson Welles': 196, 'Malcolm X': 171, 'Nick Offerman': 164, '2Pac': 155, 'Kanye West': 148, 'Robert Kardashian': 141, 'Mr. Rogers': 129, 'Norm MacDonald': 12

In [10]:
import random

# Normalise raw label strings to 0 (bonafide/real) or 1 (spoof/fake)
# ITW meta.csv uses  'bona-fide'  and  'spoof'
REAL_STRINGS = {'bona-fide', 'bonafide', 'real', 'genuine', '0', 0}
FAKE_STRINGS = {'spoof', 'fake', 'synthetic', '1', 1}

def label_to_int(v):
    if v is None:
        return None
    if isinstance(v, int):
        return v
    vl = str(v).lower().strip()
    if vl in REAL_STRINGS: return 0
    if vl in FAKE_STRINGS: return 1
    raise ValueError(f'Unknown label: {v!r}  — add it to REAL_STRINGS or FAKE_STRINGS above.')

int_labels = [label_to_int(l) for l in raw_label_strs]

random.seed(42)
real_idx = [i for i, l in enumerate(int_labels) if l == 0]
fake_idx = [i for i, l in enumerate(int_labels) if l == 1]
print(f'Available  real: {len(real_idx)}  fake: {len(fake_idx)}')

assert len(real_idx) >= N_PER_CLASS, f'Need {N_PER_CLASS} real, only {len(real_idx)} available'
assert len(fake_idx) >= N_PER_CLASS, f'Need {N_PER_CLASS} fake, only {len(fake_idx)} available'

sel_real     = random.sample(real_idx, N_PER_CLASS)
sel_fake     = random.sample(fake_idx, N_PER_CLASS)
selected_idx = sorted(sel_real + sel_fake)
balanced_labels = [int_labels[i] for i in selected_idx]

ds_balanced = ds_itw_split.select(selected_idx)
print(f'\nBalanced subset: {len(ds_balanced)} ({N_PER_CLASS} real + {N_PER_CLASS} fake)')
print('Label check:', Counter(balanced_labels))


ValueError: Unknown label: 'Alec Guinness'  — add it to REAL_STRINGS or FAKE_STRINGS above.

## 2. Dataset wrapper + dataloader

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from loader import _crop_policy, TARGET_SR

class ITWDataset(Dataset):
    """In-the-Wild wrapper. real/bonafide=0, spoof/fake=1."""
    def __init__(self, hf_split, int_labels, mode='eval'):
        self.ds     = hf_split
        self.labels = int_labels
        self.mode   = mode

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        ex  = self.ds[idx]
        wav = torch.tensor(ex['audio']['array'], dtype=torch.float32).unsqueeze(0)
        wav = _crop_policy(wav, self.mode)
        return {
            'audio':       wav,
            'label':       torch.tensor(self.labels[idx]).long(),
            'sample_rate': TARGET_SR,
        }

BATCH_SIZE  = 20
NUM_WORKERS = 4

itw_dataset    = ITWDataset(ds_balanced, balanced_labels, mode='eval')
val_dataloader = DataLoader(
    itw_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
)
print(f'Dataset: {len(itw_dataset)} samples | {len(val_dataloader)} batches')


## 3. Load model

In [ ]:
from phoneme_GAT.modules import Phoneme_GAT_lit

CKPT_PATH = 'robust_goat.ckpt'  # swap to mixed_goat.ckpt / goat.ckpt to compare

cfg = Namespace(
    PhonemeGAT=Namespace(
        backbone='wavlm', use_raw=False, use_GAT=True,
        n_edges=10, use_aug=True, use_pool=True, use_clip=True,
    )
)

model = Phoneme_GAT_lit.load_from_checkpoint(CKPT_PATH, cfg=cfg)
model = model.cuda()
torch.set_float32_matmul_precision('medium')
model.eval()
print(f'Loaded {CKPT_PATH} on', next(model.parameters()).device)

## 4. Augmentation helpers

In [ ]:
import math, numpy as np
import torchaudio
import torchaudio.functional as AF
from fractions import Fraction

SR = 16_000

def _match_len(x, n):
    if x.shape[-1] < n:
        x = torch.nn.functional.pad(x, (0, n - x.shape[-1]))
    return x[..., :n]

def aug_pitch_up(w, semitones):
    f = Fraction(2 ** (semitones / 12)).limit_denominator(20)
    return _match_len(AF.resample(w, f.numerator, f.denominator), w.shape[-1])

def aug_pitch_down(w, semitones):
    f = Fraction(2 ** (-semitones / 12)).limit_denominator(20)
    return _match_len(AF.resample(w, f.numerator, f.denominator), w.shape[-1])

def aug_phase_noise(w, noise_level):
    spec  = torch.fft.rfft(w)
    phase = torch.angle(spec) + torch.randn_like(torch.angle(spec)) * noise_level * math.pi
    return torch.fft.irfft(torch.abs(spec) * torch.exp(1j * phase), n=w.shape[-1])

def aug_amp_scale(w, scale):   return w * scale
def aug_volume(w, gain_db):    return w * (10 ** (gain_db / 20))

def aug_additive_noise(w, snr_db):
    sp = w.pow(2).mean().clamp(min=1e-9)
    n  = torch.randn_like(w)
    np_ = n.pow(2).mean().clamp(min=1e-9)
    return w + (sp / np_ / 10 ** (snr_db / 10)).sqrt() * n

def aug_time_stretch(w, rate):
    f = Fraction(rate).limit_denominator(20)
    return _match_len(AF.resample(w, f.denominator, f.numerator), w.shape[-1])

def aug_lowpass(w, cutoff_hz):  return AF.lowpass_biquad(w, SR, cutoff_frequency=cutoff_hz)
def aug_highpass(w, cutoff_hz): return AF.highpass_biquad(w, SR, cutoff_frequency=cutoff_hz)

def aug_reverberation(w, t60=0.5, room_scale=0.5):
    decay  = math.exp(-6.91 / (t60 * SR))
    ir_len = int(t60 * SR * room_scale)
    ir = torch.zeros(1, 1, ir_len)
    ir[0, 0, 0] = 1.0
    for i in range(1, ir_len):
        ir[0, 0, i] = ir[0, 0, i-1] * decay + torch.randn(1).item() * (1 - decay) * 0.1
    ir = ir / ir.abs().max().clamp(min=1e-9)
    out = torch.nn.functional.conv1d(
        w.view(1, 1, -1), ir, padding=ir_len - 1
    )[..., :w.shape[-1]]
    return out.view_as(w)

def aug_codec_mulaw(w):
    return torchaudio.functional.mu_law_decoding(
        torchaudio.functional.mu_law_encoding(w.clamp(-1, 1), 255), 255)

def build_aug_fn(name, params):
    fns = {
        'pitch_up':       lambda w: aug_pitch_up(w, **params),
        'pitch_down':     lambda w: aug_pitch_down(w, **params),
        'phase_noise':    lambda w: aug_phase_noise(w, **params),
        'amp_scale':      lambda w: aug_amp_scale(w, **params),
        'volume':         lambda w: aug_volume(w, **params),
        'additive_noise': lambda w: aug_additive_noise(w, **params),
        'time_stretch':   lambda w: aug_time_stretch(w, **params),
        'lowpass':        lambda w: aug_lowpass(w, **params),
        'highpass':       lambda w: aug_highpass(w, **params),
        'reverberation':  lambda w: aug_reverberation(w, **params),
        'codec_mulaw':    lambda w: aug_codec_mulaw(w),
    }
    if name not in fns:
        raise ValueError(f'Unknown aug: {name}')
    return fns[name]

print('Augmentation helpers ready.')

## 5. Single-augmentation smoke-test (optional)

In [ ]:
import wandb
from pytorch_lightning import Trainer
from pytorch_lightning.loggers import WandbLogger
from callbacks_rational import (
    BinaryACC_Callback, BinaryAUC_Callback, EER_Callback,
    TPR_Callback, TNR_Callback, FPR_Callback, FNR_Callback,
)

ACTIVE_AUG = 'reverberation'  # change to any aug name to smoke-test
AUG_PARAMS = {
    'pitch_up':       {'semitones': 2},
    'pitch_down':     {'semitones': 2},
    'phase_noise':    {'noise_level': 0.012},
    'amp_scale':      {'scale': 1.20},
    'volume':         {'gain_db': 4.0},
    'additive_noise': {'snr_db': 32},
    'time_stretch':   {'rate': 1.07},
    'lowpass':        {'cutoff_hz': 6500},
    'highpass':       {'cutoff_hz': 120},
    'reverberation':  {'t60': 0.6, 'room_scale': 0.5},
    'codec_mulaw':    {},
}

class AugmentedDataset(Dataset):
    def __init__(self, base, aug_fn):
        self.base   = base
        self.aug_fn = aug_fn
    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        item = self.base[idx]
        wav  = item['audio']
        item['audio'] = self.aug_fn(wav.unsqueeze(0)).squeeze(0)
        return item

aug_fn = build_aug_fn(ACTIVE_AUG, AUG_PARAMS[ACTIVE_AUG])
aug_dl = DataLoader(AugmentedDataset(itw_dataset, aug_fn),
                    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

if wandb.run is not None: wandb.finish()

trainer = Trainer(
    accelerator='gpu', devices=1,
    logger=WandbLogger(
        project='DeepfakeDetectionRenewed', entity='krishrawat0222-f',
        name=f'itw_smoke_{ACTIVE_AUG}', log_model=False,
        tags=['smoke', 'in-the-wild', ACTIVE_AUG],
    ),
    callbacks=[
        BinaryACC_Callback(batch_key='label', output_key='logit'),
        BinaryAUC_Callback(batch_key='label', output_key='logit'),
        EER_Callback(batch_key='label', output_key='logit'),
        TPR_Callback(batch_key='label', output_key='logit'),
        TNR_Callback(batch_key='label', output_key='logit'),
        FPR_Callback(batch_key='label', output_key='logit'),
        FNR_Callback(batch_key='label', output_key='logit'),
    ],
)

res = trainer.validate(model=model, dataloaders=aug_dl)
print(f'[{ACTIVE_AUG}]', res)
if wandb.run is not None: wandb.finish()

## 6. Full augmentation sweep (all attacks x 5 param values)

In [ ]:
import csv, json

ATTACK_SWEEPS = {
    'pitch_up':       [{'semitones': v} for v in [1, 2, 3, 4, 5]],
    'pitch_down':     [{'semitones': v} for v in [1, 2, 3, 4, 5]],
    'phase_noise':    [{'noise_level': v} for v in [0.003, 0.006, 0.012, 0.020, 0.030]],
    'amp_scale':      [{'scale': v} for v in [0.80, 0.90, 1.00, 1.10, 1.20]],
    'volume':         [{'gain_db': v} for v in [-6.0, -3.0, 0.0, 3.0, 6.0]],
    'additive_noise': [{'snr_db': v} for v in [40, 32, 24, 16, 8]],
    'time_stretch':   [{'rate': v} for v in [0.75, 0.9, 1.07, 1.25, 1.5]],
    'lowpass':        [{'cutoff_hz': v} for v in [7000, 5000, 3500, 2500, 1500]],
    'highpass':       [{'cutoff_hz': v} for v in [40, 80, 120, 200, 350]],
    'reverberation':  [
        {'t60': 0.20, 'room_scale': 0.25},
        {'t60': 0.35, 'room_scale': 0.40},
        {'t60': 0.50, 'room_scale': 0.50},
        {'t60': 0.70, 'room_scale': 0.65},
        {'t60': 0.90, 'room_scale': 0.80},
    ],
    'codec_mulaw':    [{}],
}

SKIP_AUGS   = set()   # e.g. {'pitch_up', 'codec_mulaw'}
RESULTS_CSV = 'itw_augmentation_attack_results.csv'
FIELDNAMES  = ['aug_name', 'params', 'acc', 'auc', 'eer', 'tpr', 'tnr', 'fpr', 'fnr']

with open(RESULTS_CSV, 'w', newline='') as f:
    csv.DictWriter(f, fieldnames=FIELDNAMES).writeheader()

print(f'CSV ready: {RESULTS_CSV}')

In [ ]:
for aug_name, param_list in ATTACK_SWEEPS.items():
    if aug_name in SKIP_AUGS:
        print(f'[SKIP] {aug_name}'); continue

    for params in param_list:
        run_name = 'itw_' + aug_name + '_' + '_'.join(f'{k}{v}' for k,v in params.items())
        print(f'\n>>> {run_name}')

        aug_ds = AugmentedDataset(itw_dataset, build_aug_fn(aug_name, params))
        aug_dl = DataLoader(aug_ds, batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=NUM_WORKERS)

        if wandb.run is not None: wandb.finish()

        wlog = WandbLogger(
            project='DeepfakeDetectionRenewed', entity='krishrawat0222-f',
            name=run_name, log_model=False,
            tags=['sweep', 'in-the-wild', aug_name],
        )
        wlog.experiment.config.update({
            'dataset': 'mueller91/In-The-Wild',
            'n_real': N_PER_CLASS, 'n_fake': N_PER_CLASS,
            'aug_name': aug_name, 'aug_params': params, 'ckpt': CKPT_PATH,
        }, allow_val_change=True)

        t = Trainer(
            accelerator='gpu', devices=1, logger=wlog,
            callbacks=[
                BinaryACC_Callback(batch_key='label', output_key='logit'),
                BinaryAUC_Callback(batch_key='label', output_key='logit'),
                EER_Callback(batch_key='label', output_key='logit'),
                TPR_Callback(batch_key='label', output_key='logit'),
                TNR_Callback(batch_key='label', output_key='logit'),
                FPR_Callback(batch_key='label', output_key='logit'),
                FNR_Callback(batch_key='label', output_key='logit'),
            ],
        )
        res = t.validate(model=model, dataloaders=aug_dl)
        m   = res[0] if res else {}

        row = {'aug_name': aug_name, 'params': json.dumps(params),
               'acc': m.get('val_acc',''), 'auc': m.get('val_auc',''),
               'eer': m.get('val_eer',''), 'tpr': m.get('val_tpr',''),
               'tnr': m.get('val_tnr',''), 'fpr': m.get('val_fpr',''),
               'fnr': m.get('val_fnr','')}
        with open(RESULTS_CSV, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=FIELDNAMES).writerow(row)

        print(f"  acc={row['acc']}  auc={row['auc']}  eer={row['eer']}")
        if wandb.run is not None: wandb.finish()

print(f'\nSweep complete. Results -> {RESULTS_CSV}')

## 7. No-augmentation baseline

In [ ]:
if wandb.run is not None: wandb.finish()

bl_log = WandbLogger(
    project='DeepfakeDetectionRenewed', entity='krishrawat0222-f',
    name=f'itw_baseline_{CKPT_PATH.replace(".ckpt","")}',
    log_model=False, tags=['baseline', 'in-the-wild', 'no-aug'],
)
bl_log.experiment.config.update({
    'dataset': 'mueller91/In-The-Wild',
    'n_real': N_PER_CLASS, 'n_fake': N_PER_CLASS,
    'aug_name': 'none', 'ckpt': CKPT_PATH,
}, allow_val_change=True)

bl_trainer = Trainer(
    accelerator='gpu', devices=1, logger=bl_log,
    callbacks=[
        BinaryACC_Callback(batch_key='label', output_key='logit'),
        BinaryAUC_Callback(batch_key='label', output_key='logit'),
        EER_Callback(batch_key='label', output_key='logit'),
        TPR_Callback(batch_key='label', output_key='logit'),
        TNR_Callback(batch_key='label', output_key='logit'),
        FPR_Callback(batch_key='label', output_key='logit'),
        FNR_Callback(batch_key='label', output_key='logit'),
    ],
)

bl_res = bl_trainer.validate(model=model, dataloaders=val_dataloader)
print('ITW baseline (no aug):', bl_res)
if wandb.run is not None: wandb.finish()